In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("ExpenseMonitoringETL") \
    .getOrCreate()

In [0]:
users = [
    (1, "User1"),
    (2, "User2"),
    (3, "User3")
]

user_columns = [
    "user_id",
    "user_name"
]

user_df = spark.createDataFrame(users, user_columns)

user_df.show()

+-------+---------+
|user_id|user_name|
+-------+---------+
|      1|    User1|
|      2|    User2|
|      3|    User3|
+-------+---------+



In [0]:
expenses = [
    (1, "Food", 450, "2026-06"),
    (1, "Transport", 200, "2026-06"),
    (1, "Bills", 800, "2026-06"),
    (2, "Shopping", 1500, "2026-06"),
    (2, "Food", 350, "2026-06"),
    (3, "Entertainment", 2500, "2026-06"),
    (3, "Transport", 150, "2026-06")
]

expense_columns = [
    "user_id",
    "category",
    "amount",
    "month"
]

expense_df = spark.createDataFrame(expenses, expense_columns)

expense_df.show()

+-------+-------------+------+-------+
|user_id|     category|amount|  month|
+-------+-------------+------+-------+
|      1|         Food|   450|2026-06|
|      1|    Transport|   200|2026-06|
|      1|        Bills|   800|2026-06|
|      2|     Shopping|  1500|2026-06|
|      2|         Food|   350|2026-06|
|      3|Entertainment|  2500|2026-06|
|      3|    Transport|   150|2026-06|
+-------+-------------+------+-------+



In [0]:
combined_df = user_df.join(
    expense_df,
    "user_id"
)

combined_df.show()

+-------+---------+-------------+------+-------+
|user_id|user_name|     category|amount|  month|
+-------+---------+-------------+------+-------+
|      1|    User1|         Food|   450|2026-06|
|      1|    User1|    Transport|   200|2026-06|
|      1|    User1|        Bills|   800|2026-06|
|      2|    User2|     Shopping|  1500|2026-06|
|      2|    User2|         Food|   350|2026-06|
|      3|    User3|Entertainment|  2500|2026-06|
|      3|    User3|    Transport|   150|2026-06|
+-------+---------+-------------+------+-------+



In [0]:
from pyspark.sql.functions import sum

summary_df = combined_df.groupBy(
    "user_id",
    "user_name",
    "month"
).agg(
    sum("amount").alias("monthly_spend")
)

summary_df.show()

+-------+---------+-------+-------------+
|user_id|user_name|  month|monthly_spend|
+-------+---------+-------+-------------+
|      1|    User1|2026-06|         1450|
|      2|    User2|2026-06|         1850|
|      3|    User3|2026-06|         2650|
+-------+---------+-------+-------------+



In [0]:
from pyspark.sql.functions import lit, col

summary_df = summary_df.withColumn(
    "budget",
    lit(5000)
)

summary_df = summary_df.withColumn(
    "savings",
    col("budget") - col("monthly_spend")
)

summary_df.show()

+-------+---------+-------+-------------+------+-------+
|user_id|user_name|  month|monthly_spend|budget|savings|
+-------+---------+-------+-------------+------+-------+
|      1|    User1|2026-06|         1450|  5000|   3550|
|      2|    User2|2026-06|         1850|  5000|   3150|
|      3|    User3|2026-06|         2650|  5000|   2350|
+-------+---------+-------+-------------+------+-------+



In [0]:
from pyspark.sql.functions import when

summary_df = summary_df.withColumn(
    "alert",
    when(col("monthly_spend") > 4000, "High Spending")
    .otherwise("Normal")
)

summary_df.show()

+-------+---------+-------+-------------+------+-------+------+
|user_id|user_name|  month|monthly_spend|budget|savings| alert|
+-------+---------+-------+-------------+------+-------+------+
|      1|    User1|2026-06|         1450|  5000|   3550|Normal|
|      2|    User2|2026-06|         1850|  5000|   3150|Normal|
|      3|    User3|2026-06|         2650|  5000|   2350|Normal|
+-------+---------+-------+-------------+------+-------+------+



In [0]:
summary_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/tmp/monthly_summary_delta")

In [0]:
summary_df.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("/tmp/monthly_summary_csv")